# Sentiment Analysis 

This notebook implements sentiment analysis, to assign a numeric value to the sentiment of comments. 

In [1]:
import numpy as np
import regex

from transformers import BertTokenizerFast, pipeline

from database.comments import Comments

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks

/opt/conda/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Can use the sentiment analysis pipeline from HuggingFace

In [2]:
cs = Comments(env='dev')
df = cs.read_all()
df = df[:1000]
print('df shape:', df.shape)

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.
df shape: (1000, 13)


In [3]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text,lsoa_code
0,81088,Ealing,214950FUL_395,214950FUL,77 Perryn Road London W3 7LT W3 7LT,Objects,2021-09-14,The proposed block is not in keeping with the ...,2025-04-09,51.51392,-0.25988,The proposed block is not in keeping with the ...,E09000009
1,94784,Barnet,21/3726/FUL_242,21/3726/FUL,7 shrublands close 7 LONDON N20 9JB,Objects,2021-09-02,Barnet House come back with a bigger scheme (2...,2025-04-11,51.63047,-0.16740,come back with a bigger scheme (260 rather tha...,E09000003
2,86884,Brent,24/0661_15,24/0661,"27 Cornmow Drive, London, NW10 1BA",Objects,2024-04-24,I'm objecting to the following application num...,2025-04-10,51.55450,-0.24291,I'm objecting to the following application num...,E09000005
3,81113,Ealing,214950FUL_420,214950FUL,28 Shakespeare Road Acton London W3 6SJ W3 6SJ,Objects,2021-09-13,"My son attends Ark Byron School, who had previ...",2025-04-09,51.51047,-0.26450,"My son attends School, who had previously made...",E09000009
4,86928,Barnet,23/3246/FUL_11,23/3246/FUL,46 Pattison Road London nw2 2hj,Objects,2023-08-25,I am owner and resident of 46 Pattison Road an...,2025-04-10,51.56115,-0.19422,I am owner and resident of 46 Pattison and hav...,E09000003


In [4]:
# Split by maximum length in order to avoid truncation
nlp = NLP_Tasks()

# df = nlp.split_text_by_length(df, column='cleaned_comment_text', max_length=100, overlap=20)

Device set to use cpu
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [5]:
data = df['cleaned_comment_text'].to_list()
data_stance = df['stance'].to_list()

In [6]:
def split_period_or_length(comment, max_length=100):
    sentences = regex.split(r'(?<=[.!?]) +', comment)
    split_sentences = []
    for sentence in sentences:
        if len(sentence) > max_length:
            # Split long sentences into smaller chunks
            for i in range(0, len(sentence), max_length):
                split_sentences.append(sentence[i:i + max_length])
        else:
            split_sentences.append(sentence)
    return split_sentences

In [7]:
sentiment_model = pipeline(model="finiteautomata/bertweet-base-sentiment-analysis", device=0)

def sentiment_score(comment):
    
    # Split the comment into sentences
    # sentences = regex.split(r'(?<=[.!?]) +', comment)
    sentences = split_period_or_length(comment, max_length=100)
    n = len(sentences)

    # Analyse sentiment for each sentence
    sentiment_results = sentiment_model(sentences)

    # Calculate score by adding 'POS' scores and subtracting 'NEG' scores
    score = 0
    for result in sentiment_results:
        if result['label'] == 'POS':
            score += result['score']
        elif result['label'] == 'NEG':
            score -= result['score']

    score = float(score / n)
    return score

Device set to use cpu


In [ ]:
df['sentiment_score'] = df['cleaned_comment_text'].apply(sentiment_score)

In [12]:
df['sentiment_conflict'] = df.apply(
    lambda row: row['sentiment_score'] > 0.5 and row['stance'] == 'Objects',
    axis=1
)

In [13]:
df

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text,lsoa_code,sentiment_score,sentiment_conflict
0,81088,Ealing,214950FUL_395,214950FUL,77 Perryn Road London W3 7LT W3 7LT,Objects,2021-09-14,The proposed block is not in keeping with the ...,2025-04-09,51.513920,-0.259880,The proposed block is not in keeping with the ...,E09000009,-0.455600,False
1,94784,Barnet,21/3726/FUL_242,21/3726/FUL,7 shrublands close 7 LONDON N20 9JB,Objects,2021-09-02,Barnet House come back with a bigger scheme (2...,2025-04-11,51.630470,-0.167400,come back with a bigger scheme (260 rather tha...,E09000003,-0.454589,False
2,86884,Brent,24/0661_15,24/0661,"27 Cornmow Drive, London, NW10 1BA",Objects,2024-04-24,I'm objecting to the following application num...,2025-04-10,51.554500,-0.242910,I'm objecting to the following application num...,E09000005,-0.380429,False
3,81113,Ealing,214950FUL_420,214950FUL,28 Shakespeare Road Acton London W3 6SJ W3 6SJ,Objects,2021-09-13,"My son attends Ark Byron School, who had previ...",2025-04-09,51.510470,-0.264500,"My son attends School, who had previously made...",E09000009,-0.422725,False
4,86928,Barnet,23/3246/FUL_11,23/3246/FUL,46 Pattison Road London nw2 2hj,Objects,2023-08-25,I am owner and resident of 46 Pattison Road an...,2025-04-10,51.561150,-0.194220,I am owner and resident of 46 Pattison and hav...,E09000003,-0.159993,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,88168,Brent,22/3481_21,22/3481,"16 Pebworth Road, Harrow, HA1 3UD",Objects,2022-10-17,The council consulted on an Article 4 directio...,2025-04-10,51.568050,-0.325020,The council consulted on an Article 4 directio...,E09000005,0.000000,False
996,88178,Barnet,22/3977/FUL_6,22/3977/FUL,32 The Ridgeway London Nw11 8qs,Objects,2022-09-02,1) The 9 flats in the area is overdevelopment....,2025-04-10,51.571537,-0.201509,1) The 9 flats in the area is overdevelopment....,E09000003,-0.166391,False
997,97741,Barnet,23/4117/FUL_105,23/4117/FUL,1 Manor Road Barnet EN5 2LH,Objects,2023-12-11,I am Barnet born and bred and 64 yrs old. I am...,2025-04-11,51.647460,-0.204450,I am Barnet born and bred and 64 yrs old. I am...,E09000003,-0.400140,False
998,97749,Barnet,23/4117/FUL_113,23/4117/FUL,100 Salisbury Road Barnet EN5 4JN,Objects,2023-12-09,The green belt should be sacred. They should b...,2025-04-11,51.655520,-0.206390,The green belt should be sacred. They should b...,E09000003,-0.192788,False


In [14]:
len(df[df['sentiment_conflict']==True])

2

In [17]:
df[df['sentiment_conflict']==True]['cleaned_comment_text'].to_list()

['This little parade of shops provide an essential centre for our neighbourhood and a valuable place to shop - particularly the greengrocers and hair salon - for locals without needing to use the car. For many, the green grocers is a community hub which provides people, particularly the elder population with a friendly service and local shop unrivalled by those on and . Please reconsider building this development for the sake of the local community and environment.',
 'Much needed update to a well loved building. All my children love going to activities and hope to go for many more years. It would be sk beneficial for the area.']

In [31]:
conflict_comments = []
conflict_comments_score = []
for i in range(len(data)):
    comment = data[i]
    # print(comment)
    score = sentiment_score(comment)
    if score > 0 and data_stance[i] == 'Objects':
        conflict_comments.append(data[i])
        conflict_comments_score.append(score)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [24]:
for item in zip(conflict_comments, conflict_comments_score):
    print(f"Comment: {item[0]}\nSentiment Score: {item[1]:.4f}\n")

In [11]:
len(conflict_comments)

7

In [13]:
practise_sentence1 = "your website, the parking survey seems to be the same one that was submitted in february last year, having been carried out in the middle of the night on two days in december 2020. on that basis, the objections which i raised to the last ' outline ' application in my e - mail to your office on 7 february 2021 ( which were acknowledged by on 11 february last year ) are as valid as they were then, so i wish to resubmit those comments ( see below ). i do"
practise_sentence2 = "evidenced by the numerous accidents that have been discussed at the area committee as well as parent and child safety during the school weeks."

In [14]:
sentiment_model = pipeline(model="finiteautomata/bertweet-base-sentiment-analysis", device='cpu')

# Split the comment into sentences
split_sentences = regex.split(r'(?<=[.!?]) +', practise_sentence2)
sentiment = sentiment_model(split_sentences)

for sentence, result in zip(split_sentences, sentiment):
    print(f"Sentence: {sentence}\nSentiment: {result['label']} (Score: {result['score']:.4f})\n")

Device set to use cpu


Sentence: evidenced by the numerous accidents that have been discussed at the area committee as well as parent and child safety during the school weeks.
Sentiment: NEU (Score: 0.7089)

